# File Allocation Table (FAT)
To write files to a drive in a computer, you should first put that drive a data structure called __file system__.
Initializing a file system on a drive i.e. putting the file system structure is called __formatting__.
Then the drive is said to be __formatted (with the file system)__.

There are several file systems including FAT12, FAT16, FAT32, ExtFAT, NTFS, HPFS, ext2, ext3 etc.
The most simplest are FAT family of file systems (FAT12, FAT16, FAT32).

On FAT file systems the files are written to drive as a **linked-list** of chunks. A __chunk__ is a part the file.

## FAT12
In FAT12, four data structures are used:
- Boot Sector
- FAT Table
- Directory
- Linked List

Basically, to read a file on a drive formatted with FAT12:
1. read the **boot sector** (first 512 bytes) of the drive.
Then according to the information on the first sector:
2. read the **FAT** table.
Calculate the position of the root directory using the FAT table. With this information
3. read the **root directory**.
    - find the position of the first chunk _(head of the linked-list)_ of the file in the root directory:
4. read the first chunk of the file.
    - after finish read the position of the next chunk of the file.
    - repeat these until there is no more chunks.

## Testing

Accessing a real drive data on modern operating systems requires elevated permissions.
Instead, we will use a file with a FAT file system structure on it.
In attachment you will find an image of a floppy drive formatted with FAT12.
Open that file in binary mode as follows:

In [1]:
image = open('sample.img','rb')

## Parsing The Boot Sector

Read the boot sector (first sector) by:

In [2]:
bs = image.read(512)

You can use `struct` module of python to parse the structures.
To use that:

In [3]:
import struct

Boot sector structure is parsed by the following:

In [4]:
(
    BS_OEMName,
    BPB_BytsPerSec,
    BPB_SecPerClus,
    BPB_RsvdSecCnt,
    BPB_NumFATs,
    BPB_RootEntCnt,
    BPB_TotSec16,
    BPB_Media,
    BPB_FATSz16,
    BPB_SecPerTrk,
    BPB_NumHeads,
    BPB_HiddSec,
    BPB_TotSec32,
    BS_DrvNum,
    BS_BootSig,
    BS_VolID,
    BS_VolLab,
    BS_FilSysType,
) = struct.unpack("<3x8s HBHBHHBHHHII B1xBI 11s8s", bs[:62])

In [5]:
print(
    f"BS_OEMName:\t{BS_OEMName.decode()}",
    f"BPB_BytsPerSec:\t{BPB_BytsPerSec}",
    f"BPB_SecPerClus:\t{BPB_SecPerClus}",
    f"BPB_RsvdSecCnt:\t{BPB_RsvdSecCnt}",
    f"BPB_NumFATs:\t{BPB_NumFATs}",
    f"BPB_RootEntCnt:\t{BPB_RootEntCnt}",
    f"BPB_TotSec16:\t{BPB_TotSec16}",
    f"BPB_Media:\t{BPB_Media}",
    f"BPB_FATSz16:\t{BPB_FATSz16}",
    f"BPB_SecPerTrk:\t{BPB_SecPerTrk}",
    f"BPB_NumHeads:\t{BPB_NumHeads}",
    f"BPB_HiddSec:\t{BPB_HiddSec}",
    f"BPB_TotSec32:\t{BPB_TotSec32}",
    f"BS_DrvNum:\t{BS_DrvNum}",
    f"BS_BootSig:\t{BS_BootSig}",
    f"BS_VolID:\t{BS_VolID}",
    f"BS_VolLab:\t{BS_VolLab.decode()}",
    f"BS_FilSysType:\t{BS_FilSysType.decode()}",
    sep="\n",
)

BS_OEMName:	WINIMAGE
BPB_BytsPerSec:	512
BPB_SecPerClus:	1
BPB_RsvdSecCnt:	1
BPB_NumFATs:	2
BPB_RootEntCnt:	224
BPB_TotSec16:	2880
BPB_Media:	240
BPB_FATSz16:	9
BPB_SecPerTrk:	18
BPB_NumHeads:	2
BPB_HiddSec:	0
BPB_TotSec32:	0
BS_DrvNum:	0
BS_BootSig:	41
BS_VolID:	1100881118
BS_VolLab:	PYTHON     
BS_FilSysType:	FAT12   


# Assignment:
There is a file `ASSIGNMENT.TXT` in the root directory of file system image `sample.img`.
Read that file by implementing the remaining of this notebook using the Python programming language.
Add nessesary explanations as markdown. 

## Step 1: Find and Read the Root Directory

First, we need to calculate where the file list (the root directory) starts and how big it is. We'll use the `BPB_RsvdSecCnt`, `BPB_NumFATs`, `BPB_FATSz16`, and `BPB_RootEntCnt` values from the Boot Sector. The root directory is located right after the FAT tables.

In [6]:
# --- Step 1: Find and Read the Root Directory ---

# Calculate the starting SECTOR number of the Root Directory
root_dir_start_sector = BPB_RsvdSecCnt + (BPB_NumFATs * BPB_FATSz16)

# Calculate the starting BYTE location
root_dir_start_byte = root_dir_start_sector * BPB_BytsPerSec

# Calculate the total size of the Root Directory in bytes
# (Max entries * 32 bytes per entry)
root_dir_size_bytes = BPB_RootEntCnt * 32

print(f"Root Directory starts at byte: {root_dir_start_byte}")
print(f"Root Directory size in bytes: {root_dir_size_bytes}")

# Go to (seek) that location in the image file
image.seek(root_dir_start_byte)

# Read the entire root directory
root_dir_data = image.read(root_dir_size_bytes)

Root Directory starts at byte: 9728
Root Directory size in bytes: 7168


## Step 2: Search for the File in the Root Directory

Now we have all the file records in the `root_dir_data` variable. We will loop through this data in 32-byte chunks to find the entry for the file named `ASSIGNMENTTXT`.
When we find it, we'll save its first cluster number (Bytes 26-27) and its file size (Bytes 28-31).

In [7]:
# --- Step 2: Find the file alias AUTOMATICALLY ---

print("--- Starting Step 2: Automatic File Search ---")

file_found = False
first_cluster = 0
file_size = 0

# This is the attribute for a Long File Name (LFN) entry
ATTR_LONG_NAME = 0x0F

# Loop through each 32-byte entry
for i in range(BPB_RootEntCnt):
    entry = root_dir_data[i*32 : (i+1)*32]
    first_byte = entry[0]

    # 0x00 = End of directory, stop
    if first_byte == 0x00:
        break
    
    # 0xE5 = Deleted entry, skip
    if first_byte == 0xE5:
        continue
        
    # Get the attribute byte
    attr = entry[11]
    
    # If it's an LFN entry, skip it
    if attr == ATTR_LONG_NAME:
        continue
        
    # --- This is a normal short name entry ---
    name = entry[0:11]
    
    # Check if the name starts with "ASSIGN"
    if name.startswith(b"ASSIGN"):
        print(f"File alias found! Using: {name}")
        
        # This is our file! Get its info.
        first_cluster = struct.unpack("<H", entry[26:28])[0]
        file_size = struct.unpack("<I", entry[28:32])[0]
        
        file_found = True
        print(f"File's first cluster: {first_cluster}")
        print(f"File's size: {file_size} bytes")
        break # Found it, stop searching

if not file_found:
    print("Error: Could not find any file starting with 'ASSIGN'.")

print("--- Finished Step 2 ---")

--- Starting Step 2: Automatic File Search ---
File alias found! Using: b'ASSIGN~0TXT'
File's first cluster: 377
File's size: 13 bytes
--- Finished Step 2 ---


## Step 3: Read the FAT (File Allocation Table)

We found the first piece of the file, but what if the file is split into multiple pieces? The FAT (File Allocation Table) tells us. 
This table is a "map" that shows which cluster links to the next cluster. We need to read this map from the disk and store it in a `fat_table` variable.

In [8]:
# --- Step 3: Read the FAT (File Allocation Table) ---

# Calculate the starting BYTE location of the FAT
fat_start_byte = BPB_RsvdSecCnt * BPB_BytsPerSec

# Calculate the total SIZE of the FAT
fat_size_bytes = BPB_FATSz16 * BPB_BytsPerSec

print(f"FAT table starts at byte: {fat_start_byte}")

# Go to that location
image.seek(fat_start_byte)

# Read the entire FAT table
fat_table = image.read(fat_size_bytes)

print("--- Finished Step 3: FAT Table is now in memory ---")

FAT table starts at byte: 512
--- Finished Step 3: FAT Table is now in memory ---


## Step 4: Follow the Cluster Chain to Read the File

This is the final and most important step.

1.  First, we'll calculate the start of the 'Data Area'. This is where the actual file data begins.
2.  We will write a special helper function (`get_next_cluster_from_fat12`) to read FAT12's 1.5-byte (12-bit) entries.
3.  Finally, we will follow the chain. We start at `first_cluster`, read its data, ask the `fat_table` (our map) for the *next* cluster, read its data, and so on. We'll combine all these pieces until we hit the End of Chain (EOC) marker.

In [9]:
# --- Step 4: Find Data Area and Read File ---

# 1. Calculate where the Data Area starts
# (Right after the root directory)
data_area_start_byte = root_dir_start_byte + root_dir_size_bytes

# 2. Calculate the size of one cluster
cluster_size_bytes = BPB_SecPerClus * BPB_BytsPerSec

print(f"Data area (clusters) starts at byte: {data_area_start_byte}")
print(f"Each cluster is {cluster_size_bytes} bytes")

# Helper function to read the 12-bit (1.5 byte) FAT12 entries
def get_next_cluster_from_fat12(fat_data, cluster_num):
    # Calculate the byte offset (multiply by 1.5)
    offset = int(cluster_num * 1.5)
    
    # Read 2 bytes (a WORD) from that offset
    word = struct.unpack("<H", fat_data[offset : offset+2])[0]
    
    if cluster_num % 2 == 0:
        # If CLUSTER NUMBER IS EVEN, get the lower 12 bits
        return word & 0x0FFF
    else:
        # If CLUSTER NUMBER IS ODD, get the upper 12 bits
        return word >> 4

# --- File Reading Loop ---

file_content = b"" # We will build the file content here
current_cluster = first_cluster

# Keep reading until we have the whole file (file_size)
while file_size > 0:
    
    # 1. Calculate the actual location of this cluster on disk
    # (We subtract 2 because clusters start at 2, not 0)
    cluster_location = data_area_start_byte + (current_cluster - 2) * cluster_size_bytes
    
    # 2. Go there and read the cluster's data
    image.seek(cluster_location)
    data_chunk = image.read(cluster_size_bytes)
    
    # 3. Add the data to our file content
    bytes_to_read = min(len(data_chunk), file_size)
    file_content += data_chunk[:bytes_to_read]
    
    # 4. Decrease the remaining file size
    file_size -= bytes_to_read
    
    # 5. Get the next cluster number from the FAT map
    next_cluster = get_next_cluster_from_fat12(fat_table, current_cluster)
    
    # 6. Check for End of Chain (EOC)
    # For FAT12, EOC is >= 0x0FF8
    if next_cluster >= 0xFF8:
        break # End of file
    
    # Continue the loop with the next cluster
    current_cluster = next_cluster

# --- RESULT ---
print("\n--- ASSIGNMENT.TXT CONTENT ---")
print(file_content.decode('ascii'))
print("-----------------------------------")

Data area (clusters) starts at byte: 16896
Each cluster is 512 bytes

--- ASSIGNMENT.TXT CONTENT ---
Hello, World
-----------------------------------


In [10]:
# --- Final Step: Close ---
# We are done, so we close the image file.
image.close()